# Banner to prefilled DonSheet

This notebook uses the same prefill routine as the Streamlit **Karen → PreFilling the DonSheet with Banner Data** workflow. Keeping that routine in one place means the notebook and Streamlit always apply the same Banner normalization, department routing, sorting, template formatting, date handling, and worksheet-protection behavior.

The workflow uses the newest formula-enabled DonSheet in `0_source_files/default_DonSheet`, places the output in `0_DonSheet`, and leaves it unprotected unless protection is explicitly enabled below.

In [ ]:
from pathlib import Path
import sys

# Resolve the repository location whether this notebook is launched from its
# own directory or from the project root.
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'jupyter_notebooks' else NOTEBOOK_DIR
PROGRAMS_DIR = PROJECT_ROOT / 'programs'
if str(PROGRAMS_DIR) not in sys.path:
    sys.path.insert(0, str(PROGRAMS_DIR))

BANNER_DIR = PROJECT_ROOT / '0_BannerSheets'
OUTPUT_DIR = PROJECT_ROOT / '0_DonSheet'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The notebook expects exactly one Banner file directly in 0_BannerSheets.
# This is the notebook equivalent of selecting one uploaded Banner file in Streamlit.
banner_candidates = sorted(
    (path for path in BANNER_DIR.iterdir() if path.suffix.lower() in {'.csv', '.xls', '.xlsx'}),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not banner_candidates:
    raise FileNotFoundError(f'No CSV, XLS, or XLSX Banner file found in {BANNER_DIR}')
if len(banner_candidates) != 1:
    raise RuntimeError(
        'Keep only the Banner workbook to import directly in ' + str(BANNER_DIR) + ': ' +
        str([path.name for path in banner_candidates])
    )
BANNER_PATH = banner_candidates[0]

# This matches Streamlit's optional "Password-protect the prefilled DonSheet" choice.
# False creates an unprotected workbook. When True, set a non-empty password.
APPLY_SHEET_PROTECTION = False
SHEET_PASSWORD = ''

if APPLY_SHEET_PROTECTION and not SHEET_PASSWORD:
    raise ValueError('Enter SHEET_PASSWORD when APPLY_SHEET_PROTECTION is True.')

print(f'Banner source: {BANNER_PATH.name}')
print('Worksheet protection: ' + ('enabled' if APPLY_SHEET_PROTECTION else 'disabled'))


In [ ]:
from app.prefill_donsheet_from_banner import prefill_donsheet_from_banner

# This is the exact function called by Streamlit. It selects the latest
# WithFormulas template, routes every 700–799 course exclusively to DMIN,
# sorts each imported tab by Subject, Course Number, Course Section, and CRN
# sorted, writes true Excel dates using d-mmm-yy, and preserves template styles
# and formulas.
result = prefill_donsheet_from_banner(
    BANNER_PATH.read_bytes(),
    BANNER_PATH.name,
    worksheet_password=SHEET_PASSWORD if APPLY_SHEET_PROTECTION else None,
)

output_path = OUTPUT_DIR / result.output_filename
output_path.write_bytes(result.excel_bytes)

print(f'Template used: {result.template_filename}')
print(f'Created: {output_path}')
print('Rows imported by tab:')
for sheet_name, row_count in result.rows_by_department.items():
    print(f'  {sheet_name}: {row_count}')


## What this follows from Streamlit

- The newest `WithFormulas` DonSheet template is selected automatically.
- Courses numbered 700–799 are sent only to `DMIN`; they are excluded from every other tab.
- Rows in every imported tab are sorted by Subject, Course Number, Course Section, then CRN sorted.
- New `Course Start Date` and `Course End Date` template headers are supported, with a fallback for older Intensive Period headers.
- Existing template cell formatting (including text columns, header styling, and `d-mmm-yy` date display) and formulas are retained.
- Workbook protection is optional and uses the configured password only when enabled.